In [85]:
import requests
import time

In [86]:
import os
from dotenv import load_dotenv

load_dotenv()
key = os.getenv("key")
endpoint="https://part1.cognitiveservices.azure.com/"

In [87]:
url = f"{endpoint}formrecognizer/documentModels/prebuilt-document:analyze?api-version=2023-07-31"

In [88]:
headers = {
    "Ocp-Apim-Subscription-Key": key,
    "Content-Type": "application/octet-stream"
}

In [89]:
with open("image.png", "rb") as f:
    response = requests.post(url, headers=headers, data=f)

In [90]:
print("Status:", response.status_code)

Status: 202


In [95]:
if response.status_code == 202:
    operation_url = response.headers["Operation-Location"]

    while True:
        result = requests.get(operation_url, headers={
            "Ocp-Apim-Subscription-Key": key
        })

        result_json = result.json()

        if result_json["status"] == "succeeded":
            print("Extraction Completed")
            break
        elif result_json["status"] == "failed":
            print("Failed")
            break

        time.sleep(2)
else:
    print("Error:", response.text)

Extraction Completed


In [94]:
text = result_json["analyzeResult"]["content"]
print("\n--- EXTRACTED FIELDS ---\n")
lines = text.split("\n")

for i, line in enumerate(lines):
    if "Full Name" in line:
        print("Name:", lines[i+1])
    elif "Policy Number" in line:
        print("Policy Number:", lines[i+1])
    elif "Claim Amount" in line:
        print("Claim Amount:", lines[i+1].replace("₹", "").strip())
    elif "Date of Incident" in line:
        print("Date:", lines[i+1])


--- EXTRACTED FIELDS ---

Policy Number: P12345
Name: Rahul Sharma
Date: 05-02-2024
Claim Amount: 70,000
